# Teste técnico do TCC — LLM no Colab, v3 (sem rótulos)

Plano da segunda revisão do Fable. **Nenhuma métrica aqui usa o rótulo**: os pares do conjunto de desenvolvimento são lidos sem a coluna de verdade, para não gastar o desenvolvimento antes da hora.

Mede, para cada configuração (modelo × quantização × tokens de resposta):
1. **Massa absoluta** que o modelo coloca nos dois tokens de resposta (deve ficar perto de 1).
2. **Concordância entre o primeiro token e o texto gerado** com 5 tokens.
3. **Sensibilidade à ordem**: taxa de inversão de sinal entre A×B e B×A, e direção média (B×A − A×B), com IC por bootstrap.
4. **Prior do token**: escore numa entrada sem conteúdo ("N/A" × "N/A"), como em Calibrate Before Use.
5. Tempo por par.

**Antes:** Ambiente de execução → T4 GPU. Faça upload do arquivo **pares_dev_v0_sem_rotulo.csv** no painel de arquivos (ícone de pasta) e depois **Executar tudo**. Leva de 15 a 30 minutos.

In [ ]:
!pip -q install bitsandbytes
import torch, time, math, statistics, csv, random, platform, gc
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
assert torch.cuda.is_available(), "Ative a GPU T4 antes de rodar"
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())
random.seed(42)

In [ ]:
MAX_PARES = 300   # pares do arquivo usados por configuração
with open("pares_dev_v0_sem_rotulo.csv", encoding="utf-8") as f:
    PARES = [(r["descricao_a"], r["descricao_b"]) for r in csv.DictReader(f)]
random.shuffle(PARES)
PARES = PARES[:MAX_PARES]
print(len(PARES), "pares")

REGRA = ("Você compara descrições de produtos de cupons fiscais de supermercados brasileiros. "
         "Duas descrições são o MESMO produto somente se tiverem a mesma marca, a mesma variante "
         "(tipo, sabor, versão) e o mesmo conteúdo (peso ou volume). ")
PROMPTS = {
    "1/0": (REGRA + "Responda apenas com um dígito: 1 se forem o mesmo produto, 0 se forem produtos diferentes.", "1", "0"),
    "sim/não": (REGRA + "Responda apenas com uma palavra: Sim se forem o mesmo produto, Não se forem produtos diferentes.", "Sim", "Não"),
}

CONFIGS = [  # (nome, repositório, 4 bits?)
    ("3B-fp16", "Qwen/Qwen2.5-3B-Instruct", False),
    ("3B-4bit", "Qwen/Qwen2.5-3B-Instruct", True),
    ("7B-4bit", "Qwen/Qwen2.5-7B-Instruct", True),
]

In [ ]:
def carregar(repo, q4):
    tok = AutoTokenizer.from_pretrained(repo)
    if q4:
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        m = AutoModelForCausalLM.from_pretrained(repo, quantization_config=cfg, device_map="cuda", torch_dtype=torch.float16)
    else:
        m = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16).to("cuda")
    return tok, m.eval()

def id_unico(tok, palavra):
    ids = tok.encode(palavra, add_special_tokens=False)
    return ids[0], len(ids)   # se a palavra tiver mais de um token, usamos o primeiro e registramos isso

def entrada(tok, sistema, a, b):
    msgs = [{"role": "system", "content": sistema},
            {"role": "user", "content": f"Descrição A: {a}\nDescrição B: {b}\nResposta:"}]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(texto, return_tensors="pt").to("cuda")

def ler(tok, m, sistema, id_pos, id_neg, a, b, gerar=False):
    e = entrada(tok, sistema, a, b)
    with torch.no_grad():
        logits = m(**e).logits[0, -1].float()
    lp = torch.log_softmax(logits, -1)
    r = {"logodds": (lp[id_pos] - lp[id_neg]).item(),
         "massa": (lp[id_pos].exp() + lp[id_neg].exp()).item(),
         "top": tok.decode([int(logits.argmax())]),
         "dtype_logits": str(m(**e).logits.dtype) if gerar else ""}
    if gerar:
        with torch.no_grad():
            out = m.generate(**e, max_new_tokens=5, do_sample=False)
        r["texto"] = tok.decode(out[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return r

def ic_boot(valores, f=statistics.mean, n=2000):
    est = [f(random.choices(valores, k=len(valores))) for _ in range(n)]
    est.sort()
    return f(valores), est[int(0.025 * n)], est[int(0.975 * n)]

In [ ]:
resultados, resumo = [], []
for nome, repo, q4 in CONFIGS:
    t0 = time.perf_counter()
    tok, m = carregar(repo, q4)
    print(f"\n=== {nome}: carregado em {time.perf_counter()-t0:.0f}s")
    for pnome, (sistema, pos, neg) in PROMPTS.items():
        id_pos, n_pos = id_unico(tok, pos); id_neg, n_neg = id_unico(tok, neg)
        prior = ler(tok, m, sistema, id_pos, id_neg, "N/A", "N/A")["logodds"]
        inv, dif, massas, concord, tempos = [], [], [], [], []
        for i, (a, b) in enumerate(PARES):
            t = time.perf_counter()
            ab = ler(tok, m, sistema, id_pos, id_neg, a, b, gerar=(i < 30))
            ba = ler(tok, m, sistema, id_pos, id_neg, b, a)
            tempos.append(time.perf_counter() - t)
            if i == 0:
                dt = ab["dtype_logits"]
            inv.append(1 if (ab["logodds"] > 0) != (ba["logodds"] > 0) else 0)
            dif.append(ba["logodds"] - ab["logodds"])
            massas += [ab["massa"], ba["massa"]]
            if i < 30:
                gerado_pos = ab["texto"].lower().startswith(pos.lower())
                concord.append(1 if gerado_pos == (ab["logodds"] > 0) else 0)
            resultados.append({"config": nome, "prompt": pnome, "descricao_a": a, "descricao_b": b,
                               "logodds_AB": ab["logodds"], "logodds_BA": ba["logodds"],
                               "massa_AB": ab["massa"], "massa_BA": ba["massa"],
                               "texto_gerado_AB": ab.get("texto", "")})
        taxa, ti, ts = ic_boot(inv)
        dmed, di, ds = ic_boot(dif)
        linha = {"config": nome, "prompt": pnome, "tokens_pos/neg": f"{n_pos}/{n_neg}",
                 "dtype_logits": dt,
                 "massa_mediana": statistics.median(massas), "massa_min": min(massas),
                 "concord_1o_token_texto": statistics.mean(concord),
                 "taxa_inversao": taxa, "inv_ic": (ti, ts), "direcao_media": dmed, "dir_ic": (di, ds),
                 "prior_NA": prior, "seg_por_par": statistics.median(tempos)}
        resumo.append(linha)
        print(f"{nome:8} {pnome:7} | massa med {linha['massa_mediana']:.3f} min {linha['massa_min']:.3f} | "
              f"1º token×texto {linha['concord_1o_token_texto']:.2f} | inversão {taxa:.3f} [{ti:.3f}; {ts:.3f}] | "
              f"direção {dmed:+.2f} [{di:+.2f}; {ds:+.2f}] | prior N/A {prior:+.2f} | {linha['seg_por_par']:.2f}s/par")
    del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
print("Legenda: tokens_pos/neg = quantos tokens tem cada resposta (1/1 é o ideal).")
for l in resumo:
    print(l["config"], l["prompt"], "tokens", l["tokens_pos/neg"], "| dtype dos logits:", l["dtype_logits"] or "n/d")
with open("resultado_v3_detalhe.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(resultados[0].keys())); w.writeheader(); w.writerows(resultados)
with open("resultado_v3_resumo.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(resumo[0].keys())); w.writeheader(); w.writerows(resumo)
print("Salvos: resultado_v3_resumo.csv e resultado_v3_detalhe.csv (baixe os dois pelo painel de arquivos)")